In [1]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

In [2]:
pwd

'/Users/minhtamninale/Documents/Angela/Spots_troubleshooting/3rd_647_as_PPE/Post_processing'

In [4]:
df = pd.DataFrame()

folder_path = '../PPE_late'

dfs = []

for filename in os.listdir(folder_path):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, sep ='\s+', header=None, dtype={3: str}, names=['x', 'y', 'z', 'conformation'])
        dfs.append(df)
        
df = pd.concat(dfs, ignore_index=True)

file_paths =glob.glob('../PPE_late/*.txt')

for file_path in file_paths:
    file_data = pd.read_csv(file_path, sep='\s+', header=None, dtype={3: str}, names=['x', 'y', 'z', 'conformation'])
    
      
    file_name = file_path.split('/')[-1]
    number = file_name.split('_')[0]
    number = int(number)
    
    file_data['Embryo_ID'] = number
    
    df = pd.concat([df,file_data],ignore_index=True)
    
    main_frame = df.dropna()

In [5]:
main_frame

,x,y,z,conformation,Embryo_ID
2342,39.086144,6.767449,0.72,111,2.0
2343,72.758329,18.882833,3.24,011,2.0
2344,49.715991,49.055752,3.24,111,2.0
2345,75.696392,33.177006,4.92,111,2.0
2346,73.616640,64.736426,4.56,111,2.0
...,...,...,...,...,...
4679,32.285683,30.734122,5.40,110,5.0
4680,56.021273,44.533115,6.48,000,5.0
4681,43.740829,73.187484,4.08,011,5.0
4682,69.127015,43.047578,6.84,100,5.0


In [6]:
def conformation_meaning(conformation): 
    meanings = { 
        '000': 'Not_touching',
        '100': '3_prime_touch_PPE',
        '010':'5_prime_touch_PPE',
        '001':'3_prime_touch_5_prime',
        '110':'PPE_prime_middle',
        '011':'5_middle',
        '101':'3_prime_middle',
        '111':'All_touching'
    }
    if conformation in meanings: 
        return meanings[conformation]
    else:
        return 'Not supported'
    
main_frame.loc[:,'Meaning'] = main_frame['conformation'].apply(conformation_meaning)

/var/folders/6g/g2plc5bd3jv_sqpbk23jqwm40000gn/T/ipykernel_14317/1598569033.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_frame.loc[:,'Meaning'] = main_frame['conformation'].apply(conformation_meaning)


In [7]:
main_frame.head()

,x,y,z,conformation,Embryo_ID,Meaning
2342,39.086144,6.767449,0.72,111,2.0,All_touching
2343,72.758329,18.882833,3.24,011,2.0,5_middle
2344,49.715991,49.055752,3.24,111,2.0,All_touching
2345,75.696392,33.177006,4.92,111,2.0,All_touching
2346,73.616640,64.736426,4.56,111,2.0,All_touching


In [8]:
def count_conformation_per_embryo(df):
    counts = {}
    embryo_ids = main_frame['Embryo_ID'].unique()
    for embryo_id in embryo_ids:
        embryo_df = main_frame.loc[main_frame['Embryo_ID'] == embryo_id]
        count_000 = len(embryo_df.loc[embryo_df["conformation"] == '000'])
        count_100 = len(embryo_df.loc[embryo_df["conformation"] == '100'])
        count_010 = len(embryo_df.loc[embryo_df["conformation"] == '010'])
        count_001 = len(embryo_df.loc[embryo_df["conformation"] == '001'])
        count_110 = len(embryo_df.loc[embryo_df["conformation"] == '110'])
        count_011 = len(embryo_df.loc[embryo_df["conformation"] == '011'])
        count_101 = len(embryo_df.loc[embryo_df["conformation"] == '101'])
        count_111 = len(embryo_df.loc[embryo_df["conformation"] == '111'])
        
        counts[embryo_id] = {
            'Counts_of_not_touching':count_000, 
            'Counts_3_prime_touch_PPE': count_100,
            'Counts_5_prime_touch_PPE': count_010,
            'Counts_3_prime_touch_5_prime':count_001,
            'Counts_PPE_prime_middle:':count_110,
            'Counts_5_middle':count_011,
            'Counts_3_prime_middle':count_101,
            'Counts_all_touching':count_111
        }
        
    counts_df = pd.DataFrame(counts).T.reset_index()
    counts_df.columns = ["Embryo_ID", "Counts_of_not_touching","Counts_3_prime_touch_5_prime", 
                        "Counts_5_prime_touch_PPE","Counts_3_prime_touch_PPE","Counts_5_prime_middle",
                        "Counts_PPE_middle","Counts_3_prime_middle","Counts_all_touching"]
    
    return counts_df

In [9]:
counts_conformation = count_conformation_per_embryo(main_frame).sort_values("Embryo_ID")
counts_conformation

,Embryo_ID,Counts_of_not_touching,Counts_3_prime_touch_5_prime,Counts_5_prime_touch_PPE,Counts_3_prime_touch_PPE,Counts_5_prime_middle,Counts_PPE_middle,Counts_3_prime_middle,Counts_all_touching
18,1.0,10,7,20,16,1,2,0,47
0,2.0,3,2,11,8,0,4,0,76
13,3.0,2,2,6,4,1,2,0,37
2,4.0,0,2,12,5,0,7,1,29
24,5.0,2,4,8,11,3,6,0,56
15,6.0,7,2,14,17,3,1,2,43
9,7.0,5,7,11,16,3,2,1,49
3,43.0,18,8,13,18,3,5,0,70
19,44.0,7,5,16,8,3,5,0,38
8,45.0,32,11,18,9,1,0,0,27


In [10]:
counts_conformation.to_csv('../PPE_late/PPE_conformation_conformation_count.csv')

In [11]:
def calculate_category_percentages(dataframe):
    #Exclude Embryo ID in calculation
    columns_to_calculate = dataframe.columns[1:]
    
    # Calculate the sum of each row
    row_sums = dataframe[columns_to_calculate].sum(axis=1)

    # Calculate the percentage of each column for each row by dividing the value of each column by the row sum and multiplying by 100
    category_percentages = dataframe[columns_to_calculate].div(row_sums, axis=0) *100
    
    category_percentages.insert(0, 'Embryo_ID', dataframe['Embryo_ID'])
    return category_percentages

In [12]:
percentage_df = calculate_category_percentages(counts_conformation)

In [13]:
percentage_df.to_csv('../PPE_late/PPE_percentages_per_embryo.csv')

In [14]:
transposed_df=counts_conformation.T
transposed_df

,18,0,13,2,24,15,9,3,19,8,...,5,6,1,7,16,11,22,23,10,17
Embryo_ID,1.0,2.0,3.0,4.0,5.0,6.0,7.0,43.0,44.0,45.0,...,52.0,53.0,54.0,55.0,66.0,67.0,68.0,69.0,70.0,71.0
Counts_of_not_touching,10.0,3.0,2.0,0.0,2.0,7.0,5.0,18.0,7.0,32.0,...,22.0,24.0,3.0,14.0,18.0,4.0,0.0,4.0,3.0,2.0
Counts_3_prime_touch_5_prime,7.0,2.0,2.0,2.0,4.0,2.0,7.0,8.0,5.0,11.0,...,5.0,13.0,3.0,8.0,14.0,2.0,0.0,8.0,5.0,0.0
Counts_5_prime_touch_PPE,20.0,11.0,6.0,12.0,8.0,14.0,11.0,13.0,16.0,18.0,...,8.0,17.0,7.0,12.0,33.0,8.0,1.0,9.0,1.0,9.0
Counts_3_prime_touch_PPE,16.0,8.0,4.0,5.0,11.0,17.0,16.0,18.0,8.0,9.0,...,8.0,43.0,5.0,14.0,29.0,8.0,2.0,10.0,4.0,4.0
Counts_5_prime_middle,1.0,0.0,1.0,0.0,3.0,3.0,3.0,3.0,3.0,1.0,...,1.0,1.0,2.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0
Counts_PPE_middle,2.0,4.0,2.0,7.0,6.0,1.0,2.0,5.0,5.0,0.0,...,5.0,3.0,2.0,6.0,5.0,1.0,0.0,1.0,0.0,1.0
Counts_3_prime_middle,0.0,0.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,...,0.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
Counts_all_touching,47.0,76.0,37.0,29.0,56.0,43.0,49.0,70.0,38.0,27.0,...,14.0,39.0,19.0,39.0,68.0,40.0,2.0,13.0,10.0,21.0


In [15]:
transposed_df.to_csv('../PPE_late/PPE_late_transformed.csv')

In [16]:
def calculate_category_sum(dataframe): 
    categories = ['Embryo_ID','Counts_of_not_touching', 'Counts_3_prime_touch_5_prime', 'Counts_5_prime_touch_PPE',
                  'Counts_3_prime_touch_PPE', 'Counts_5_prime_middle', 'Counts_PPE_middle',
                  'Counts_3_prime_middle', 'Counts_all_touching']    
    row_sums = []
    for _, row in dataframe.iterrows():
        row_sum = row.sum()
        row_sums.append(row_sum)
    row_sums_df = pd.DataFrame({'Category': categories, 'Category_Sum':row_sums})
    return row_sums_df

In [17]:
sum_category = calculate_category_sum(transposed_df)
PPE_sum = sum_category.iloc[1:]
PPE_sum

,Category,Category_Sum
1,Counts_of_not_touching,354.0
2,Counts_3_prime_touch_5_prime,161.0
3,Counts_5_prime_touch_PPE,392.0
4,Counts_3_prime_touch_PPE,330.0
5,Counts_5_prime_middle,36.0
6,Counts_PPE_middle,80.0
7,Counts_3_prime_middle,11.0
8,Counts_all_touching,978.0


In [18]:
PPE_sum.to_csv('../PPE_late/PPE_sum.csv')

In [20]:
transposed_percentage= percentage_df.T
transposed_percentage

,18,0,13,2,24,15,9,3,19,8,...,5,6,1,7,16,11,22,23,10,17
Embryo_ID,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,43.000000,44.000000,45.000000,...,52.000000,53.000000,54.000000,55.000000,66.000000,67.000000,68.0,69.000000,70.000000,71.000000
Counts_of_not_touching,9.708738,2.884615,3.703704,0.000000,2.222222,7.865169,5.319149,13.333333,8.536585,32.653061,...,34.920635,17.142857,6.976744,15.053763,10.465116,6.349206,0.0,8.888889,12.500000,5.263158
Counts_3_prime_touch_5_prime,6.796117,1.923077,3.703704,3.571429,4.444444,2.247191,7.446809,5.925926,6.097561,11.224490,...,7.936508,9.285714,6.976744,8.602151,8.139535,3.174603,0.0,17.777778,20.833333,0.000000
Counts_5_prime_touch_PPE,19.417476,10.576923,11.111111,21.428571,8.888889,15.730337,11.702128,9.629630,19.512195,18.367347,...,12.698413,12.142857,16.279070,12.903226,19.186047,12.698413,20.0,20.000000,4.166667,23.684211
Counts_3_prime_touch_PPE,15.533981,7.692308,7.407407,8.928571,12.222222,19.101124,17.021277,13.333333,9.756098,9.183673,...,12.698413,30.714286,11.627907,15.053763,16.860465,12.698413,40.0,22.222222,16.666667,10.526316
Counts_5_prime_middle,0.970874,0.000000,1.851852,0.000000,3.333333,3.370787,3.191489,2.222222,3.658537,1.020408,...,1.587302,0.714286,4.651163,0.000000,1.162791,0.000000,0.0,0.000000,4.166667,2.631579
Counts_PPE_middle,1.941748,3.846154,3.703704,12.500000,6.666667,1.123596,2.127660,3.703704,6.097561,0.000000,...,7.936508,2.142857,4.651163,6.451613,2.906977,1.587302,0.0,2.222222,0.000000,2.631579
Counts_3_prime_middle,0.000000,0.000000,0.000000,1.785714,0.000000,2.247191,1.063830,0.000000,0.000000,0.000000,...,0.000000,0.000000,4.651163,0.000000,1.744186,0.000000,0.0,0.000000,0.000000,0.000000
Counts_all_touching,45.631068,73.076923,68.518519,51.785714,62.222222,48.314607,52.127660,51.851852,46.341463,27.551020,...,22.222222,27.857143,44.186047,41.935484,39.534884,63.492063,40.0,28.888889,41.666667,55.263158


In [21]:
transposed_percentage.to_csv('../PPE_late/PPE_percentages.csv')

In [22]:
def calculate_column_medians(counts_conformation):
    columns_medians = counts_conformation.median()
    columns_median_df = pd.DataFrame(columns_medians, columns=['Median'])
    columns_median_df.reset_index(inplace=True)
    columns_median_df.rename(columns={'index': 'Categories'}, inplace=True)
    return columns_median_df

df_PPE_median = calculate_column_medians(counts_conformation)
PPE_embryo_median = df_PPE_median.drop([0]).reset_index(drop = True)

In [23]:
PPE_embryo_median

,Categories,Median
0,Counts_of_not_touching,7.0
1,Counts_3_prime_touch_5_prime,5.0
2,Counts_5_prime_touch_PPE,12.0
3,Counts_3_prime_touch_PPE,10.0
4,Counts_5_prime_middle,1.0
5,Counts_PPE_middle,2.0
6,Counts_3_prime_middle,0.0
7,Counts_all_touching,38.0


In [24]:
PPE_embryo_median.to_csv("../PPE_late/PPE_embryo_median_conformation_count.csv")